# Silver: harmonização das 3 edições num schema único

**O desafio desta camada**: cada edição da pesquisa usa um schema de colunas
diferente. Por exemplo, a edição 2023-2024 tem cabeçalhos no formato
`('P1_b ', 'Genero')` (tupla Python serializada como texto), enquanto as
edições 2024-2025 e 2025-2026 usam `1.b_genero`. Além disso, campos booleanos
são representados como `'0'/'1'` em duas edições e como `'TRUE'/'FALSE'` em
outra. Isso foi validado localmente em `src/silver_harmonization.py` (pandas);
este notebook replica exatamente a mesma lógica em PySpark.

Resultado: duas tabelas Silver.
- `tb_silver_respondentes`: uma linha por respondente, ~28 campos
  padronizados, união das 3 edições (schema largo).
- `tb_silver_tecnologias_adocao`: uma linha por `(respondente, tecnologia)`
  (schema longo/despivotado). Em vez de manter dezenas de colunas 0/1 por
  linguagem/banco/cloud/ferramenta de BI, cada tecnologia marcada como "usa"
  vira uma linha. Isso simplifica muito os `GROUP BY` da camada Gold e do
  Athena (ex: "ranking de linguagens mais usadas" é um `GROUP BY tecnologia`
  direto, em vez de somar 15 colunas manualmente).

In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5


In [ ]:
%%configure
{
  "--enable-glue-datacatalog": "true"
}


In [ ]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
import pyspark.sql.functions as F
import pyspark.sql.types as T

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)


In [ ]:
# =============================================================================
# CONFIGURACAO: valores conferidos ao vivo no console AWS do time em
# 2026-10-18 (Glue Data Catalog + S3). Troque so se os nomes mudarem.
#
# Os 3 CSVs brutos ficam juntos, soltos (sem subpasta por edicao), direto em
# s3://tech-challenge-gt26/bronze/, com nomes de arquivo que nao indicam a
# edicao de forma confiavel. O mapeamento abaixo foi confirmado por inspecao
# de conteudo (S3 Select: formato do cabecalho + pergunta "passou por layoff
# em <ano>"):
#   - "State_of_data_2022.csv"                      -> edicao 2023-2024
#   - "State_of_data_BR_2023_Kaggle - df_survey_2023.csv" -> edicao 2024-2025
#   - "Final Dataset - State of Data 2025-2026 - Kaggle.csv" -> edicao 2025-2026
#
# Os nomes de tabela abaixo (tb_bronze_*, tb_silver_*, tb_gold_*) usam o
# prefixo "tb_" de proposito, para conviver no mesmo database
# db_techchallenge_gt_fiap com as tabelas "bronze"/"silver"/"gold" que ja
# existem. Nada aqui sobrescreve ou apaga o que ja foi feito.
# =============================================================================
BUCKET = "s3://tech-challenge-gt26"

RAW_PATHS = {
    "2023-2024": f"{BUCKET}/bronze/State_of_data_2022.csv",
    "2024-2025": f"{BUCKET}/bronze/State_of_data_BR_2023_Kaggle - df_survey_2023.csv",
    "2025-2026": f"{BUCKET}/bronze/Final Dataset - State of Data 2025-2026 - Kaggle.csv",
}

# Prefixo novo e distinto do "bronze/" (lowercase) onde os CSVs brutos ja
# estao, para evitar qualquer risco de sobrescrever a fonte original.
BRONZE_PATH = f"{BUCKET}/bronze_parquet/"
SILVER_PATH = f"{BUCKET}/Silver/"
GOLD_PATH = f"{BUCKET}/Gold/"

DB_BRONZE = "db_techchallenge_gt_fiap"
DB_SILVER = "db_techchallenge_gt_fiap"
DB_GOLD = "db_techchallenge_gt_fiap"

for db in (DB_BRONZE, DB_SILVER, DB_GOLD):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")


## Mapeamento de colunas por edição

Os índices abaixo foram determinados inspecionando o header de cada CSV (ver
`docs/_headers_*.txt` e `docs/_diagnostico_categorias.txt` no protótipo local)
e validados comparando distribuições de categorias entre as 3 edições.

In [ ]:
# ATENCAO: indices abaixo re-mapeados em 2026-10-19 depois de inspecionar os
# cabecalhos reais dos 3 CSVs no S3 (via S3 Select), nao mais os CSVs de
# referencia locais. Achado ao vivo: o arquivo de "2023-2024" do time
# (State_of_data_2022.csv, 353 colunas) e uma revisao mais antiga do Kaggle
# que nao tem as perguntas de IA Generativa nem "ferramenta de BI preferida".
# Por isso esses 3 campos ficam de fora do dict dessa edicao (viram NULL
# automaticamente no unionByName(allowMissingColumns=True) mais abaixo, o que
# e o resultado correto: a pergunta nunca existiu pra essa edicao). Ja o
# arquivo real de "2024-2025" (df_survey_2023.csv, 399 colunas) tem a mesma
# estrutura que o CSV de referencia local de 2023-2024 usado antes. Por isso
# os indices que antes estavam em "2023-2024" foram movidos pra "2024-2025".
# "2025-2026" nao mudou (confirmado igual ao arquivo real, 388 colunas).
SCALAR_FIELDS = {
    "2023-2024": {
        "id_respondente": 0, "idade": 1, "faixa_idade": 2, "genero": 3,
        "cor_raca_etnia": 4, "pcd": 5, "vive_no_brasil": 8, "uf": 10,
        "regiao": 11, "nivel_ensino": 14, "area_formacao": 15,
        "situacao_trabalho": 16, "setor": 17, "numero_funcionarios": 18,
        "gestor": 19, "cargo_atual": 21, "nivel_senioridade": 22,
        "faixa_salarial": 23, "tempo_exp_dados": 24, "tempo_exp_ti": 25,
        "satisfeito": 26, "modelo_trabalho_atual": 48, "modelo_trabalho_ideal": 49,
        "area_atuacao": 91, "linguagem_preferida": 126, "cloud_preferida": 162,
        # sem ferramenta_bi_preferida / empresa_prioriza_ia_generativa /
        # uso_ia_generativa_individual: essas perguntas nao existem no CSV
        # real desta edicao (confirmado via S3 Select).
    },
    "2024-2025": {
        "id_respondente": 0, "idade": 1, "faixa_idade": 2, "genero": 3,
        "cor_raca_etnia": 4, "pcd": 5, "vive_no_brasil": 21, "uf": 23,
        "regiao": 24, "nivel_ensino": 27, "area_formacao": 28,
        "situacao_trabalho": 29, "setor": 30, "numero_funcionarios": 31,
        "gestor": 32, "cargo_atual": 34, "nivel_senioridade": 35,
        "faixa_salarial": 36, "tempo_exp_dados": 37, "tempo_exp_ti": 38,
        "satisfeito": 39, "modelo_trabalho_atual": 62, "modelo_trabalho_ideal": 63,
        "area_atuacao": 124, "linguagem_preferida": 160, "cloud_preferida": 203,
        "ferramenta_bi_preferida": 228, "empresa_prioriza_ia_generativa": 103,
        "uso_ia_generativa_individual": 238,
    },
    "2025-2026": {
        "id_respondente": 0, "idade": 2, "faixa_idade": 3, "genero": 4,
        "cor_raca_etnia": 5, "pcd": 6, "vive_no_brasil": 22, "uf": 25,
        "regiao": 26, "nivel_ensino": 31, "area_formacao": 32,
        "situacao_trabalho": 33, "setor": 34, "numero_funcionarios": 35,
        "gestor": 36, "cargo_atual": 38, "nivel_senioridade": 39,
        "faixa_salarial": 40, "tempo_exp_dados": 41, "tempo_exp_ti": 42,
        "satisfeito": 43, "modelo_trabalho_atual": 70, "modelo_trabalho_ideal": 71,
        "area_atuacao": 133, "linguagem_preferida": 144, "cloud_preferida": 197,
        "ferramenta_bi_preferida": 217, "empresa_prioriza_ia_generativa": 112,
        "uso_ia_generativa_individual": 227,
    },
}

MULTI_SELECT_RANGES = {
    "2023-2024": {"linguagem": range(111, 125), "banco_dados": range(128, 161),
                  "cloud": range(163, 166), "ferramenta_bi": range(167, 190)},
    "2024-2025": {"linguagem": range(144, 159), "banco_dados": range(162, 195),
                  "cloud": range(196, 203), "ferramenta_bi": range(205, 228)},
    "2025-2026": {"linguagem": range(145, 155), "banco_dados": range(156, 189),
                  "cloud": range(190, 197), "ferramenta_bi": range(199, 217)},
}

ANO_PESQUISA = {"2023-2024": 2023, "2024-2025": 2024, "2025-2026": 2025}
TRUE_LIKE = ["1", "TRUE", "True", "true"]
FALSE_LIKE = ["0", "FALSE", "False", "false"]


## Funções de limpeza e recodificação (idênticas à versão pandas validada)

In [ ]:
import re

def col_by_idx(df, idx):
    # F.col(nome) sozinho (e tambem df[nome]) quebra com AnalysisException em
    # varios cabecalhos da edicao 2023-2024, serializados como tupla Python em
    # texto (ex: "('P1_b ', 'Genero')"), porque o parser de nomes qualificados
    # do Spark tenta interpretar o ponto/virgula/aspas dentro do nome. Envolver
    # em crase (`...`) forca o nome inteiro a ser tratado como um unico
    # identificador literal.
    return F.col(f"`{df.columns[idx]}`")

def to_bool01(c):
    return F.when(c.isin(TRUE_LIKE), 1).when(c.isin(FALSE_LIKE), 0).otherwise(None).cast("int")

def blank_to_null(c):
    trimmed = F.trim(c)
    return F.when(trimmed == "", None).otherwise(trimmed)

def faixa_salarial_valor_medio(c):
    # "de R$ 8.001/mês a R$ 12.000/mês" -> ponto médio; casos abertos têm valor fixo.
    low = F.regexp_extract(c, r"R\$\s*([\d.]+)/m[êe]s\s+a\s+R\$", 1)
    high = F.regexp_extract(c, r"a\s+R\$\s*([\d.]+)/m[êe]s", 1)
    low_num = F.regexp_replace(low, r"\.", "").cast("double")
    high_num = F.regexp_replace(high, r"\.", "").cast("double")
    return (
        F.when(c.isNull(), None)
        .when(c.contains("Acima de"), F.lit(45000.0))
        .when(c.contains("Menos de"), F.lit(500.0))
        .when((low_num.isNotNull()) & (high_num.isNotNull()), (low_num + high_num) / 2)
        .otherwise(None)
    )

def classificar_prioridade_ia(c):
    t = F.lower(c)
    return (
        F.when(c.isNull(), None)
        .when(t.contains("principal prioridade"), "Prioridade alta (foco executivo)")
        .when(t.contains("próximos 2-4 anos") | t.contains("proximos 2-4 anos"), "Prioridade de médio prazo")
        .when(t.contains("mais ou menos"), "Iniciativa isolada / pouco foco")
        .when(t.contains("não sei") | t.contains("nao sei"), "Não sabe opinar")
        .when(t.contains("não é uma iniciativa") | t.contains("nao e uma iniciativa"), "Não é prioridade")
        .otherwise("Outro")
    )

def classificar_uso_ia(c):
    t = F.lower(c)
    nao_usa = t.contains("não utilizo nenhum") | t.contains("nao utilizo nenhum")
    usa = F.when(c.isNull() | (c == ""), None).when(nao_usa, 0).otherwise(1)
    paga = F.when(t.contains("pagas") | t.contains("pago") | t.contains("paga pela"), 1).otherwise(0)
    codigo = F.when(t.contains("copilot") | t.contains("ai para código") | t.contains("ai para codigo"), 1).otherwise(0)
    return usa, paga, codigo

def rotulo_tecnologia(header: str) -> str:
    m = re.match(r"^\(?'?[\w.]+\s*'?,?\s*'(.+)'\)?$", header)
    if m:
        return m.group(1).strip()
    m2 = re.match(r"^\d+(?:\.[0-9a-zA-Z]+)*[_ ](.+)$", header)
    if m2:
        return m2.group(1).strip()
    # Formato real das edicoes 2024-2025/2025-2026 no S3 (diferente do CSV de
    # referencia local usado antes): prefixo de letra+numero repetido, com ou
    # sem underscore entre letra e numero (ex. "c_4_c_3_python",
    # "p4_d_1_sql") -> captura so o nome final.
    m3 = re.match(r"^[a-z]_?\d+(?:_[a-z]_?\d+)*_(.+)$", header)
    if m3:
        return m3.group(1).strip()
    return header.strip()


## Construção de `tb_silver_respondentes`

In [ ]:
CATEGORICAS = [
    "faixa_idade", "genero", "cor_raca_etnia", "pcd", "uf", "regiao",
    "nivel_ensino", "area_formacao", "situacao_trabalho", "setor",
    "numero_funcionarios", "cargo_atual", "nivel_senioridade", "faixa_salarial",
    "tempo_exp_dados", "tempo_exp_ti", "modelo_trabalho_atual",
    "modelo_trabalho_ideal", "area_atuacao", "linguagem_preferida",
    "cloud_preferida", "ferramenta_bi_preferida",
]

respondentes_frames = []
for edicao, col_map in SCALAR_FIELDS.items():
    df = bronze_dfs[edicao] if "bronze_dfs" in dir() else spark.table(
        f"{DB_BRONZE}.tb_bronze_{edicao.replace('-', '_')}"
    )
    # remove respondentes duplicados (encontrados na validacao local: a
    # propria pesquisa tem alguns tokens de respondente repetidos na origem)
    id_col_name = df.columns[col_map["id_respondente"]]
    antes = df.count()
    df = df.dropDuplicates([id_col_name])
    print(f"{edicao}: {antes - df.count()} duplicado(s) removido(s)")

    selecionadas = [col_by_idx(df, idx).alias(nome) for nome, idx in col_map.items()]
    out = df.select(*selecionadas)

    out = out.withColumn("idade", F.col("idade").cast("int"))
    out = out.withColumn(
        "idade", F.when((F.col("idade") < 14) | (F.col("idade") > 80), None).otherwise(F.col("idade"))
    )
    for c in CATEGORICAS:
        # "in out.columns", nao assumindo que toda edicao tem todo campo: a
        # 2023-2024 real nao tem, por exemplo, ferramenta_bi_preferida.
        if c in out.columns:
            out = out.withColumn(c, blank_to_null(F.col(c)))

    out = out.withColumn("vive_no_brasil", to_bool01(F.col("vive_no_brasil")))
    out = out.withColumn("gestor", to_bool01(F.col("gestor")))
    out = out.withColumn("satisfeito_empresa_atual", to_bool01(F.col("satisfeito"))).drop("satisfeito")

    out = out.withColumn("faixa_salarial_valor_medio", faixa_salarial_valor_medio(F.col("faixa_salarial")))

    # A edicao 2023-2024 (arquivo real) nao tem as perguntas de IA Generativa
    # (ver nota no mapeamento de colunas acima). Nesse caso os campos
    # derivados ficam NULL explicitamente, em vez de falhar tentando ler uma
    # coluna que nao existe no select() desta edicao.
    if "empresa_prioriza_ia_generativa" in col_map:
        out = out.withColumn("prioridade_ia_empresa", classificar_prioridade_ia(F.col("empresa_prioriza_ia_generativa")))
    else:
        out = out.withColumn("prioridade_ia_empresa", F.lit(None).cast("string"))

    if "uso_ia_generativa_individual" in col_map:
        usa, paga, codigo = classificar_uso_ia(F.col("uso_ia_generativa_individual"))
    else:
        usa = paga = codigo = F.lit(None).cast("int")
    out = out.withColumn("usa_ia_generativa", usa).withColumn("paga_por_ia_generativa", paga).withColumn("usa_ia_para_codigo", codigo)

    out = out.withColumn("edicao_pesquisa", F.lit(edicao))
    out = out.withColumn("ano_pesquisa", F.lit(ANO_PESQUISA[edicao]))
    out = out.withColumn("id_unico", F.concat(F.lit(edicao + "_"), F.col("id_respondente").cast("string")))

    respondentes_frames.append(out)

tb_silver_respondentes = respondentes_frames[0]
for f in respondentes_frames[1:]:
    tb_silver_respondentes = tb_silver_respondentes.unionByName(f, allowMissingColumns=True)

print(f"tb_silver_respondentes: {tb_silver_respondentes.count():,} linhas")
tb_silver_respondentes.groupBy("edicao_pesquisa").count().show()


In [ ]:
(
    tb_silver_respondentes.write
    .mode("overwrite")
    .format("parquet")
    .option("path", SILVER_PATH + "respondentes/")
    .partitionBy("ano_pesquisa")
    .saveAsTable(f"{DB_SILVER}.tb_silver_respondentes")
)
print(f"gravado e catalogado: {DB_SILVER}.tb_silver_respondentes")


## Construção de `tb_silver_tecnologias_adocao` (schema longo)

Para cada edição e cada categoria de tecnologia (linguagem, banco de dados,
cloud, ferramenta de BI), cada coluna checkbox (0/1) vira uma linha,
mantendo apenas as linhas onde a tecnologia foi marcada como usada
(`usa = 1`), para não inflar a tabela com zeros.

In [ ]:
tec_frames = []
for edicao, categorias in MULTI_SELECT_RANGES.items():
    df = bronze_dfs[edicao] if "bronze_dfs" in dir() else spark.table(
        f"{DB_BRONZE}.tb_bronze_{edicao.replace('-', '_')}"
    )
    id_col = df.columns[SCALAR_FIELDS[edicao]["id_respondente"]]
    df = df.dropDuplicates([id_col])  # mesmo criterio de deduplicacao da tb_silver_respondentes

    for categoria, idx_range in categorias.items():
        for idx in idx_range:
            header = df.columns[idx]
            tecnologia = rotulo_tecnologia(header)
            sub = df.select(
                F.concat(F.lit(edicao + "_"), F.col(f"`{id_col}`").cast("string")).alias("id_unico"),
                F.lit(edicao).alias("edicao_pesquisa"),
                F.lit(ANO_PESQUISA[edicao]).alias("ano_pesquisa"),
                F.lit(categoria).alias("categoria_tecnologia"),
                F.lit(tecnologia).alias("tecnologia"),
                F.col(f"`{header}`").cast("int").alias("usa"),
            ).filter(F.col("usa") == 1)
            tec_frames.append(sub)

tb_silver_tecnologias_adocao = tec_frames[0]
for f in tec_frames[1:]:
    tb_silver_tecnologias_adocao = tb_silver_tecnologias_adocao.unionByName(f)

print(f"tb_silver_tecnologias_adocao: {tb_silver_tecnologias_adocao.count():,} linhas")


In [ ]:
(
    tb_silver_tecnologias_adocao.write
    .mode("overwrite")
    .format("parquet")
    .option("path", SILVER_PATH + "tecnologias_adocao/")
    .saveAsTable(f"{DB_SILVER}.tb_silver_tecnologias_adocao")
)
print(f"gravado e catalogado: {DB_SILVER}.tb_silver_tecnologias_adocao")
